# [실습] LangChain을 이용한 데이터 생성과 처리


LCEL의 기본 문법인 Prompt | llm | Parser 구조에 대해 배웠습니다.   
이번 실습에서는 출력을 구조화하고, LLM을 연결하는 방법에 대해 알아봅니다.


### 라이브러리 설치  

랭체인 OpenAI 모듈을 설치합니다.

In [ ]:
%pip install langchain langchain_openai dotenv rich -q

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

if os.environ.get('OPENAI_API_KEY'):
    print('OpenAI API 키 확인')

### LLM 모델 불러오기

In [ ]:
from langchain.chat_models import init_chat_model

gpt_llm = init_chat_model(
    "gpt-5.2", reasoning_effort='low')

## JsonOutputParser 로 Json 형식의 출력 만들기

LLM의 출력을 구조화하면, 데이터 후처리를 하지 않고도 다른 코드와 연결할 수 있습니다.   
JSON 형식의 출력을 구성해 보겠습니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import JsonOutputParser

jsonparser = JsonOutputParser()

JSON 파서의 역할은 JSON 규격에 맞는 텍스트를 Dict 형식으로 변환하는 것으로,     
실제 형식에 대한 조건을 프롬프트로 전달해야 합니다.

In [ ]:
jsonparser.get_format_instructions()

In [ ]:
recipe_template = ChatPromptTemplate([
    ('system','당신은 전세계의 이색적인 퓨전 조리법의 전문가입니다.'),
    ('user','''저는 다음의 재료와 조건을 이용한 환상적인 퓨전 다이닝을 만들고 싶습니다. 1가지 메뉴만 추천해주세요!
레시피에 대한 정보를 JSON 형식으로 출력해주세요.

[재료]: {ingredient}
[조건]: {condition}
''')
])

recipe_chain = recipe_template | gpt_llm | jsonparser

In [ ]:
response = recipe_chain.invoke({'ingredient':'콜라, 고수', 'condition':'디저트'})
response

In [ ]:
# Dict 구조: 추출 가능
# response['menuName']

Json으로 파싱하는 방법은 활용도가 높지만, 실행할 때마다 결과뿐만 아니라 형식도 달라진다는 문제가 있습니다.

In [ ]:
response = recipe_chain.invoke({'ingredient':'문어, 피넛버터', 'condition':'메인 요리'})
response

## Pydantic을 이용해 출력 형식 지정하기

pydantic은 데이터 형식에 제약조건을 두고 이를 준수하는지 검증하는 라이브러리입니다.


In [ ]:
from pydantic import BaseModel, Field
# pydantic 연동

class Recipe(BaseModel):
    name: str = Field(description="음식 이름")
    # name: 문자열, 설명은 "음식 이름"
    difficulty: str = Field(description="만들기의 난이도")

    origin: str = Field(description="원산지")
    ingredients: list[str] = Field(description="재료")
    # ingredients: 문자열 리스트, 설명은 "재료"

    instructions: list[str] = Field(description="조리법")
    tip: str = Field(description='조리 과정 팁')


In [ ]:
parser = JsonOutputParser(pydantic_object=Recipe)

In [ ]:
print(parser.get_format_instructions())

해당 내용을 프롬프트에 포함합니다.

In [ ]:
recipe_template2 = ChatPromptTemplate([
    ('system','당신은 전세계의 이색적인 퓨전 조리법의 전문가입니다.'),
    ('user','''저는 다음의 재료를 이용한 실험적인 음식을 만들고 싶습니다. 추천해주세요!
    레시피에 대한 정보를 JSON 형식으로 출력해주세요. 결과는 한국어로 작성하세요.
재료: {ingredient}
출력 형식 조건: {instruction}''')
])

recipe_chain2 = recipe_template2 | gpt_llm | parser


In [ ]:
recipe_chain2.invoke({'ingredient':'생강', 'instruction':parser.get_format_instructions()})

partial을 통해 먼저 일부를 입력할 수도 있습니다.

In [ ]:
recipe_template2 = ChatPromptTemplate([
    ('system','당신은 전세계의 이색적인 퓨전 조리법의 전문가입니다.'),
    ('user','''저는 {ingredient}를 이용한 실험적인 음식을 만들고 싶습니다. 추천해주세요!
    레시피에 대한 정보를 JSON 형식으로 출력해주세요. 결과는 한국어로 작성하세요.
     {instruction}''')
]).partial(instruction=parser.get_format_instructions())

recipe_chain2 = recipe_template2 | gpt_llm | parser

recipe_chain2.invoke('감자')

# LangChain Structured Output
파서를 사용하지 않고, 구조화된 출력을 생성합니다.  

In [ ]:
from rich import print as rprint
structured_llm = gpt_llm.with_structured_output(Recipe)

rprint(structured_llm)

In [ ]:
recipe_template3 = ChatPromptTemplate([
    ('system','당신은 한국 전통의 재료가 가진 다양한 맛과 특성을 활용합니다.'),
    ('user','''저는 {ingredient}를 이용한 실험적인 음식을 만들고 싶습니다. 추천해주세요!''')
])

recipe_chain3 = recipe_template3 | structured_llm
response = recipe_chain3.invoke("생강")
rprint(response)

해당 출력은 Pydantic 클래스 형식으로 생성됩니다.   
with_structured_output 기능을 지원하지 않는 경우, PydanticOutputParser를 사용해야 합니다.

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser

pydantic_parser = PydanticOutputParser(pydantic_object = Recipe)

recipe_template4 =ChatPromptTemplate([
    ('system','당신은 전세계의 이색적인 퓨전 조리법의 전문가입니다.'),
    ('user','''저는 {ingredient}를 이용한 실험적인 음식을 만들고 싶습니다. 추천해주세요!
    레시피에 대한 정보를 JSON 형식으로 출력해주세요. 결과는 한국어로 작성하세요.
     {instruction}''')
])

structured_llm2 = recipe_template4.partial(instruction = pydantic_parser.get_format_instructions()) | gpt_llm | pydantic_parser

response = structured_llm2.invoke('커피')
response

## [실습] LLM으로 보고서 개요 생성 후 섹션별 글 작성하기

Structured Output 구조를 활용해, LLM이 주제에 대한 구획을 먼저 구성하고    
해당 구획을 반복문이나 Batch로 각각 입력하여 긴 글을 쓰도록 만들어 보세요.

1. with_structured_output을 통해 주제에 대한 구획 작성하는 체인 `outliner` 만들기
2. 섹션별 글 작성 체인 `writer` 만들기
3. 반복문이나 batch()를 통해 `outliner`의 결과물을 `writer`에 전달하기
4. 최종 결과물 합치기

In [ ]:
class Sections(BaseModel):
    topic: str = Field(description="글쓰기 주제")
    sections: list[str] = Field(description="주제에 대한 세부 섹션 개요 리스트 (최대 5개 섹션)")

In [ ]:
outliner = gpt_llm.with_structured_output(Sections)
outline = outliner.invoke("""
멀티모달 LLM의 발전 과정에 대한 보고서 개요를 써줘.
각각의 개요는 병렬적 작성이 가능하도록 독립적인 내용을 담아야 하고.
개요만 보고도 내용이 구체적으로 드러나야 해.
""")
outline

In [ ]:
writer_prompt = ChatPromptTemplate([
    ('human','''보고서 주제에 대해, 하나의 섹션에 대한 전문적인 글을 작성하세요.

제목은 ##, 소목차는 ###으로 쓰고, 이외의 목차 형식은 넣지 마세요.
챕터명에 숫자를 넣지 마세요.
내용은 '입니다' 와 같은 말투로 작성하세요.

---
보고서 전체 주제: {topic}
세부 섹션 주제: {section}
''')
])
writer = writer_prompt | gpt_llm | StrOutputParser()
writer.invoke({'topic':outline.topic, 'section':outline.sections[0]})

In [ ]:
writer = writer_prompt.partial(topic=outline.topic) | gpt_llm | StrOutputParser()
# topic을 미리 채워 매개변수 1개
result = writer.batch(outline.sections)
result

In [ ]:
draft = '\n\n'.join(result)
with open('result.md', 'w', encoding='utf-8') as f:
    f.write(draft)

print(draft[0:100])

<br><br>
## Runnables

LangChain 체인의 기본 구조는 `RunnableSequence` 클래스로 구성됩니다.   

이 때, 시퀀스를 구성한 llm, prompt, chain 각 모듈은 Runnables에 해당합니다.   
Runnables은 자유롭게 체인에 포함되어 결과를 연결할 수 있습니다.



이번에는, 데이터 흐름을 제어하는 특별한 Runnable인   
RunnablePassthrough와 RunnableParallel을 이용해 체인을 구성해 보겠습니다.


<br><br>
### RunnablePassthrough
RunnablePassthrough는 체인의 직전 출력을 그대로 가져옵니다.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

prompt1 = ChatPromptTemplate(["{director}의 대표 작품은 무엇입니까? 하나의 작품만 선택하고, 해당 작품에 대해 20자 이내로 설명하세요."])
chain1 = (
    prompt1
    | gpt_llm
    | StrOutputParser()
    | {'answer': RunnablePassthrough()})

response = chain1.invoke("봉준호")
response

<br><br>
### RunnableParallel

RunnableParallel은 서로 다른 체인을 병렬적으로 실행하여 dict 구조로 전달합니다.

In [ ]:
from langchain_core.runnables import RunnableParallel

prompt1 = ChatPromptTemplate(["색깔을 하나 알려주세요, 색깔만 출력하세요."])
prompt2 = ChatPromptTemplate(["음식을 하나 알려주세요, 음식만 출력하세요."])

chain1 = prompt1 | gpt_llm | StrOutputParser()
chain2 = prompt2 | gpt_llm | StrOutputParser()

chain3 = RunnableParallel(color = chain1, food = chain2)

chain3.invoke({})

## Assign()

RunnableParallel을 사용하면 중간 체인의 결과를 전달하여, 다음 체인의 결과를 함께 얻을 수 있습니다.   

In [ ]:
prompt1 = ChatPromptTemplate(["잭슨빌은 어느 나라의 도시입니까?"])
prompt2 = ChatPromptTemplate(
    ["{country}의 대표적인 인물 3명을 나열하세요. 인물의 이름만 출력하세요."]
)

chain1 = prompt1 | gpt_llm | StrOutputParser()
chain2 = prompt2 | gpt_llm | StrOutputParser()

chain3 = RunnableParallel(country = chain1).assign(people = chain2)

chain3.invoke({})

<br><br><br><br><br><br><br><br>
chain2에서 새로운 매개변수가 추가되는 경우는 어떻게 해야 할까요?

In [ ]:
prompt1 = ChatPromptTemplate(["{city}는 어느 나라의 도시인가요? 나라 이름만 출력하세요."])
prompt2 = ChatPromptTemplate(["{country}의 유명한 인물은 누가 있나요? {num} 명의 이름을 나열하세요. 사람 이름만 ,로 구분하여 나열하세요."])

chain1 = prompt1 | gpt_llm | StrOutputParser()

chain2 = (
    RunnablePassthrough.assign(country = chain1)
    # 입력받은 city, num에 country를 추가하여 전달

    | prompt2
    # country, num을 받아 실행
    | gpt_llm
    | StrOutputParser()
)

print(chain2.invoke({"city": "잭슨빌", "num": "3"}))

<br><br>
assign을 여러 개 연결할 수 있습니다.

In [ ]:
chain4 = (prompt2
    | gpt_llm
    | StrOutputParser())

chain3 = RunnablePassthrough.assign(country = chain1).assign(res = chain4)

chain3.invoke({"city": "부에노스 아이레스", "num": "3"})

<br><br><br>JsonOutputParser를 쓴다면 아래와 같이 만들 수도 있습니다.

In [ ]:
prompt1 = ChatPromptTemplate(
    ["영화 배우 한명과 대표작 하나를 출력하세요. json 형식으로 출력하고, 각 항목은 actor, movie로 표시하세요."])
prompt2 = ChatPromptTemplate(["{actor}는 {movie}에서 어떤 역할을 했습니까?"])

chain1 = prompt1 | gpt_llm | JsonOutputParser()
chain2 =(
     chain1 | prompt2 | gpt_llm | StrOutputParser()
)
chain2.invoke({})

In [ ]:
chain3 = prompt2 | gpt_llm | StrOutputParser()

chain4 = chain1.assign(result = chain3)

chain4.invoke({})